# Exercise 9.3: Autoencoder for anomalous events

From *Programming in High Energy Particle Physics*, Chapter 9

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch09/ex09_3_solution.ipynb)

Exercise 9.3 Autoencoder for anomalous events Train an autoencoder on QCD multijet events and use it to detect anomalous events (e.g., containing a resonance). Plot the reconstruction error distribution for background and signal events, and compute the ROC curve.

<details><summary>Hint</summary>

Start from Listing 9.6 . Use plt.hist(..., log=True) for the error distributions and sklearn.metrics.roc_curve(labels, scores) with the reconstruction error as the score. Then check the caveat from Section 9.5 : plot the mean error of background events as a function of jet mass or \(p_T\). If it rises, a cut on the error will sculpt these distributions.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q numpy matplotlib scikit-learn")

**Data:** seeded synthetic QCD-like events and injected outliers. The scores are demonstration values, not an experimental sensitivity.

### Step 1: Make QCD-like and anomalous toy events

In [ ]:
import numpy as np, torch
rng=np.random.default_rng(42); torch.manual_seed(42); torch.set_num_threads(2)
W=rng.normal(size=(8,20)); W[:,0]=np.linspace(.2,1.5,8)
def sample(n,anomaly=False):
    z=rng.normal(size=(n,8)); x=z@W+rng.normal(0,.4,(n,20))
    mass=80+12*z[:,0]+rng.normal(0,4,n)
    if anomaly: x[:,8:12]+=rng.normal(4,.6,(n,4)); mass+=40
    return x,mass
train,_=sample(4000); bkg,m_b=sample(1500); sig,m_s=sample(1500,True)

### Step 2: Train on background only and calculate scores

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve,roc_auc_score
from torch import nn
scaler=StandardScaler().fit(train)
A=torch.tensor(scaler.transform(train),dtype=torch.float32)
net=nn.Sequential(nn.Linear(20,32),nn.ReLU(),nn.Linear(32,8),nn.ReLU(),nn.Linear(8,32),nn.ReLU(),nn.Linear(32,20))
opt=torch.optim.Adam(net.parameters(),lr=.005)
for epoch in range(35):
    for batch in A.split(256):
        loss=((net(batch)-batch)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
net.eval()
def error(x):
    a=torch.tensor(scaler.transform(x),dtype=torch.float32)
    with torch.no_grad(): return ((net(a)-a)**2).mean(1).numpy()
eb,es=error(bkg),error(sig); labels=np.r_[np.zeros(len(eb)),np.ones(len(es))]; scores=np.r_[eb,es]
fpr,tpr,_=roc_curve(labels,scores); print('Test AUC',round(roc_auc_score(labels,scores),3))

### Step 3: Plot ROC and check mass dependence

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,3,figsize=(14,4))
ax[0].hist(eb,bins=40,alpha=.7,label='QCD-like toy'); ax[0].hist(es,bins=40,alpha=.6,label='anomaly toy'); ax[0].set(xlabel='Mean squared reconstruction error',ylabel='Events / bin',yscale='log'); ax[0].legend()
ax[1].plot(fpr,tpr); ax[1].plot([0,1],[0,1],'--'); ax[1].set(xlabel='False positive rate',ylabel='True positive rate')
edges=np.linspace(np.percentile(m_b,1),np.percentile(m_b,99),13)
means=[eb[(m_b>=lo)&(m_b<hi)].mean() for lo,hi in zip(edges[:-1],edges[1:])]
ax[2].plot((edges[:-1]+edges[1:])/2,means,'o-'); ax[2].set(xlabel='Toy jet mass [GeV]',ylabel='Mean background error')
plt.tight_layout(); plt.show()

The anomaly score separates the injected outliers in this controlled toy. A trend in the background error versus mass warns that a score cut can sculpt the mass spectrum.